# StandUp4AI Benchmark: Download → Extract → Evaluate

**Goal:** Beat StandUp4AI's F1=0.51 baseline with our F0/spectral features.

**Data:** 3,751 comedy videos, 7 languages, 330 hours.

**All processing on Google Drive — no local disk needed.**

In [ ]:
# Cell 1: Setup
import os, sys, json, subprocess, time, glob
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/standup4ai')
BASE.mkdir(exist_ok=True)
(BASE / 'audio').mkdir(exist_ok=True)
(BASE / 'features').mkdir(exist_ok=True)

subprocess.run(['pip', 'install', '-q', 'yt-dlp', 'librosa', 'soundfile'])
import librosa, numpy as np, pandas as pd
from tqdm.auto import tqdm

print(f'✅ Ready. Base: {BASE}')

In [ ]:
# Cell 2: Clone StandUp4AI repo (small text files only)
repo = BASE / 'seq-Standup4AI'
if not repo.exists():
    os.chdir(BASE)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/sofia-callejas/seq-Standup4AI.git'], check=True)
    print(f'✅ Cloned to {repo}')
else:
    print(f'✅ Repo exists')

partition = pd.read_csv(repo / 'partition.csv')
print(f'\n{len(partition)} videos | {partition["part"].value_counts().to_dict()}')
print(f'Languages: {partition["lan"].nunique()}')

# Language to directory mapping
LANG_DIRS = {'en_us':'en_us','en_uk':'en_uk','fr':'fr','fr_ca':'fr_ca',
             'es':'es','es_ch':'es_ch','es_latam':'es_latam','pt':'pt',
             'it':'it','cs':'cs','hu':'hu'}

In [ ]:
# Cell 3: Load laughter labels
# Format: laughter_detection/EMNLP/{lang}/{video_id}.csv
# Columns: t0, t1, source, label (risa=no_risa)

def load_labels(video_id, lang):
    """Load laughter intervals for a video."""
    lang_dir = LANG_DIRS.get(lang, lang)
    csv_path = repo / 'laughter_detection' / 'EMNLP' / lang_dir / f'{video_id}.csv'
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path)
    # risa = laughter, no_risa = no laughter
    df['label_bin'] = (df['label'] == 'risa').astype(int)
    return df

# Test on sample
test_vids = partition[partition['part']=='test']['fn'].tolist()
sample = test_vids[0]
sample_lang = partition[partition['fn']==sample]['lan'].values[0]
labels = load_labels(sample, sample_lang)
if labels is not None:
    print(f'Sample {sample} ({sample_lang}): {len(labels)} segments')
    print(labels.head(10))
    print(f'\nPositive rate: {labels["label_bin"].mean():.1%}')
else:
    print(f'No labels for {sample}')
    # Count how many test videos have labels
    found = 0
    for _, row in partition[partition['part']=='test'].iterrows():
        if load_labels(row['fn'], row['lan']) is not None:
            found += 1
    print(f'Found labels for {found}/{len(test_vids)} test videos')

In [ ]:
# Cell 4: Download test audio from YouTube (Colab has no rate limit)
import yt_dlp

ydl_opts = {
    'format': 'bestaudio/best',
    'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'm4a'}],
    'outtmpl': str(BASE / 'audio' / '%(id)s.%(ext)s'),
    'quiet': True, 'no_warnings': True,
}

ok, fail = 0, 0
for i, row in tqdm(partition[partition['part']=='test'].iterrows(), total=100, desc='Downloading'):
    vid = row['fn']
    audio = BASE / 'audio' / f'{vid}.m4a'
    if audio.exists():
        ok += 1; continue
    # Check if labels exist
    if load_labels(vid, row['lan']) is None:
        continue
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([f'https://youtube.com/watch?v={vid}'])
        ok += 1
    except:
        fail += 1
    if (ok+fail) % 20 == 0:
        print(f'  OK={ok} FAIL={fail}')

print(f'\n✅ Downloaded: {ok} | Failed: {fail}')

In [ ]:
# Cell 5: Extract features
def extract_features(audio_path, t0, t1, sr=22050):
    """Extract 12-dim spectral features for segment [t0, t1]."""
    try:
        dur = min(t1 - t0, 10.0)
        if dur < 0.1:
            return None
        y, sr = librosa.load(audio_path, sr=sr, offset=t0, duration=dur)
        if len(y) < sr * 0.1:
            return None
        rms = librosa.feature.rms(y=y, hop_length=512)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]
        cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=512)[0]
        bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=512)[0]
        flat = librosa.feature.spectral_flatness(y=y, hop_length=512)[0]
        return np.array([
            np.mean(rms), np.std(rms), np.max(rms),
            np.mean(zcr), np.std(zcr),
            np.mean(cent), np.std(cent),
            np.mean(bw), np.std(bw),
            np.mean(flat), np.std(flat),
            len(y)/sr,  # duration
        ])
    except:
        return None

X_all, y_all, vids_all, langs_all = [], [], [], []

for _, row in tqdm(partition[partition['part']=='test'].iterrows(), total=100, desc='Extracting'):
    vid, lang = row['fn'], row['lan']
    audio = BASE / 'audio' / f'{vid}.m4a'
    if not audio.exists():
        continue
    labels = load_labels(vid, lang)
    if labels is None:
        continue
    for _, seg in labels.iterrows():
        feat = extract_features(audio, float(seg['t0']), float(seg['t1']))
        if feat is not None:
            X_all.append(feat)
            y_all.append(int(seg['label_bin']))
            vids_all.append(vid)
            langs_all.append(lang)

X = np.array(X_all)
y = np.array(y_all)
videos = np.array(vids_all)
langs = np.array(langs_all)

print(f'\n=== DATASET ===')
print(f'Samples: {len(y)} | Positive: {y.sum()} ({y.mean():.1%})')
print(f'Videos: {len(set(videos))} | Languages: {len(set(langs))}')
print(f'Features: {X.shape[1]}-dim')
np.savez_compressed(BASE / 'features' / 'standup4ai.npz', X=X, y=y, videos=videos, langs=langs)

In [ ]:
# Cell 6: Evaluate (Video-Level CV)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold

models = {
    'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced'),
    'XGBoost': GradientBoostingClassifier(n_estimators=100, max_depth=3),
}

gkf = GroupKFold(n_splits=min(5, len(set(videos))))
results = {}

for name, model in models.items():
    f1s = []
    for tr, te in gkf.split(X, y, videos):
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        model.fit(Xtr, y[tr])
        pred = model.predict(Xte)
        f1s.append(f1_score(y[te], pred, zero_division=0))
    results[name] = {'mean': np.mean(f1s), 'std': np.std(f1s), 'folds': f1s}
    print(f'{name:15s} F1={np.mean(f1s):.4f} ± {np.std(f1s):.4f}')

print(f'\n{"="*50}')
print(f'StandUp4AI baseline: F1=0.51')
for name, r in sorted(results.items(), key=lambda x: -x[1]['mean']):
    print(f'{name:15s} F1={r["mean"]:.4f} ± {r["std"]:.4f} {"🏆" if r["mean"] > 0.51 else ""}')
print(f'{"="*50}')

In [ ]:
# Cell 7: Per-Language Breakdown
from collections import defaultdict

# Best model: LogReg
best_model = LogisticRegression(max_iter=1000, class_weight='balanced')
sc = StandardScaler()
Xs = sc.fit_transform(X)

# Leave-one-video-out for per-language analysis
lang_f1s = defaultdict(list)
for vid in set(videos):
    mask = videos == vid
    train = ~mask
    if len(y[mask]) == 0 or y[train].sum() == 0:
        continue
    best_model.fit(Xs[train], y[train])
    pred = best_model.predict(Xs[mask])
    lang = langs[mask][0]
    lang_f1s[lang].append(f1_score(y[mask], pred, zero_division=0))

print(f'{"Language":<12} {"F1":>8} {"Videos":>8}')
print('-'*30)
for lang, scores in sorted(lang_f1s.items()):
    print(f'{lang:<12} {np.mean(scores):>8.4f} {len(scores):>8}')

all_f1 = [s for scores in lang_f1s.values() for s in scores]
print(f'\n{"="*50}')
print(f'OVERALL: F1={np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}')
print(f'StandUp4AI baseline: F1=0.51')
print(f'{"="*50}')

# Save results
final = {
    'experiment': 'StandUp4AI F0/Spectral Evaluation',
    'n_samples': len(y), 'n_videos': len(set(videos)),
    'positive_rate': float(y.mean()),
    'features': f'{X.shape[1]}-dim spectral',
    'models': {k: {'f1': v['mean'], 'std': v['std']} for k,v in results.items()},
    'per_language': {k: float(np.mean(v)) for k,v in lang_f1s.items()},
    'standup4ai_baseline': 0.51,
}
with open(BASE / 'results.json', 'w') as f:
    json.dump(final, f, indent=2)
print(f'\n✅ Results saved to {BASE / "results.json"}')